# Supporting Information — Extended results

In [ ]:
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent / "src"))
from style import PRIMARY_COLOR, SECONDARY_COLOR, finish_axis, save_figure

In [ ]:
FREQUENCY_RESPONSE_DIR = (
    Path.home() / "Desktop" / "QMUL BME" / "TFM" / "Freq new patch" / "20260603"
)

SEARCH_ROOTS = [
    Path.home() / "TFM QMUL",
    Path.home() / "Desktop" / "QMUL BME" / "TFM",
    Path.home() / "Desktop" / "dvc" / "TFM QMUL",
]


def find_exact_file(filename):
    matches = []
    for root in SEARCH_ROOTS:
        if root.exists():
            matches.extend(root.rglob(filename))
    matches = sorted(set(matches))
    if len(matches) == 1:
        return matches[0]
    if len(matches) > 1:
        print(f"Multiple matches for {filename}:")
        for path in matches:
            print(" ", path)
    return None


SUPPORT_DATA_FILE = find_exact_file("expsupports.xlsx")
COATING_DATA_FILE = find_exact_file("comparison_20251126_forreport.xlsx") or find_exact_file(
    "comparsion_20251126_forreport.xlsx"
)

### SI-2.1 Preliminary optimisation of the experimental arrangement

#### Figure SI-3 — Maximum vibration amplitude at 40 Hz across speaker–sensor distances

In [ ]:
support_raw = pd.read_excel(SUPPORT_DATA_FILE, sheet_name="Sheet1", header=None).copy()

support_col = support_raw[0].astype("string").str.strip()

is_numeric_label = support_col.str.fullmatch(r"\d+(?:\.\d+)?", na=False)

support_col = support_col.mask(is_numeric_label, pd.NA)

support_raw[0] = support_col

support_40 = support_raw.iloc[2:, [0, 1, 10, 11]].copy()

support_40.columns = ["support", "distance_cm", "maximum_amplitude", "detected_frequency_hz"]

support_40["support"] = support_40["support"].ffill()

for column in ["distance_cm", "maximum_amplitude", "detected_frequency_hz"]:
    support_40[column] = pd.to_numeric(support_40[column], errors="coerce")

support_names = {
    "No Supporter": "Bench",
    "Black Sponge": "Black foam",
    "Black Pillow (Sandbag)": "Beanbag",
    "Yellow Pillow": "Yellow foam",
}

support_40 = support_40[support_40["support"].isin(support_names)].copy()

support_40["support"] = support_40["support"].map(support_names)

support_40 = (
    support_40.dropna(subset=["distance_cm", "maximum_amplitude"])
    .sort_values(["support", "distance_cm"])
    .reset_index(drop=True)
)

fig, ax = plt.subplots(figsize=(11.8, 4.6))

plot_styles = {
    "Bench": {"color": SECONDARY_COLOR, "linestyle": "--"},
    "Black foam": {"color": "#7B8A99", "linestyle": "-"},
    "Beanbag": {"color": PRIMARY_COLOR, "linestyle": "--"},
    "Yellow foam": {"color": "#D6A33E", "linestyle": "--"},
}

for support_name in ["Bench", "Black foam", "Beanbag", "Yellow foam"]:
    data = support_40.loc[support_40["support"] == support_name].sort_values("distance_cm")

    ax.plot(
        data["distance_cm"],
        data["maximum_amplitude"],
        marker="o",
        label=support_name,
        **plot_styles[support_name],
    )

ax.set_title(
    "Maximum vibration amplitude at 40 Hz across speaker–sensor distances",
    fontsize=13,
    fontweight="semibold",
    pad=7,
)

ax.set_xlabel("Speaker–sensor distance (cm)")
ax.set_ylabel("Maximum amplitude (a.u.)")

ax.set_xticks([0, 5, 10, 15, 20, 25])
ax.set_ylim(0, 20)

ax.legend(frameon=False, ncol=4, loc="upper center", bbox_to_anchor=(0.5, -0.18))

finish_axis(ax)

fig.tight_layout()

save_figure(fig, "figure_10_support_distance_40hz")

plt.show()

### SI-2.2 Comparison of coated and uncoated patch variants

#### Figure SI-4 — Frequency response of the coated and uncoated NilocasPatch variants

In [ ]:
def extract_first_number(value):
    if pd.isna(value):
        return np.nan

    match = re.search(r"[-+]?(?:\d*\.\d+|\d+)", str(value))
    return float(match.group()) if match else np.nan


def load_coating_sheet(workbook, sheet_name):
    raw = pd.read_excel(workbook, sheet_name=sheet_name, header=None)

    frequency_columns = []

    for column in range(2, raw.shape[1]):
        header = raw.iloc[1, column]

        if pd.isna(header):
            continue

        header_text = str(header)

        if "transmitter without support" in header_text.lower():
            continue

        frequency = extract_first_number(header_text)

        if np.isnan(frequency):
            continue

        frequency_columns.append({"column": column, "frequency_hz": frequency})

    channel_rows = raw.iloc[2:18, :].copy()

    channel_rows[0] = channel_rows[0].ffill()

    records = []

    for _, row in channel_rows.iterrows():
        channel_match = re.search(r"Channel\s+(\d+)", str(row.iloc[0]), flags=re.IGNORECASE)

        if channel_match is None:
            continue

        channel = int(channel_match.group(1))

        if channel not in range(1, 9):
            continue

        version = str(row.iloc[1]).strip()

        variant_map = {"V1": "Uncoated", "V2": "Coated"}

        if version not in variant_map:
            continue

        for item in frequency_columns:
            amplitude = extract_first_number(row.iloc[item["column"]])

            records.append(
                {
                    "sensor_type": sheet_name.upper(),
                    "channel": channel,
                    "variant": variant_map[version],
                    "frequency_hz": item["frequency_hz"],
                    "maximum_amplitude": amplitude,
                }
            )

    result = pd.DataFrame(records)

    result = result.dropna(subset=["maximum_amplitude"]).reset_index(drop=True)

    return result


acc_coating = load_coating_sheet(COATING_DATA_FILE, "acc")
mic_coating = load_coating_sheet(COATING_DATA_FILE, "mic")

coating_data = pd.concat([acc_coating, mic_coating], ignore_index=True)

coating_counts = (
    coating_data.groupby(["sensor_type", "variant", "frequency_hz"])["channel"]
    .nunique()
    .reset_index(name="n_channels")
)

assert coating_counts["n_channels"].eq(8).all(), "Some groups do not have exactly 8 channels."

coating_summary = coating_data.groupby(
    ["sensor_type", "variant", "frequency_hz"], as_index=False
).agg(
    mean_amplitude=("maximum_amplitude", "mean"),
    sd_amplitude=("maximum_amplitude", lambda values: values.std(ddof=1)),
)

fig, axes = plt.subplots(1, 2, figsize=(11.8, 4.6))

for ax, sensor_type, title in [
    (axes[0], "ACC", "(a) Accelerometers"),
    (axes[1], "MIC", "(b) Microphones"),
]:
    sensor_data = coating_summary[coating_summary["sensor_type"] == sensor_type]

    for variant, color in [("Uncoated", SECONDARY_COLOR), ("Coated", PRIMARY_COLOR)]:
        data = sensor_data[sensor_data["variant"] == variant].sort_values("frequency_hz")

        ax.errorbar(
            data["frequency_hz"],
            data["mean_amplitude"],
            yerr=data["sd_amplitude"],
            marker="o",
            linestyle="-",
            color=color,
            linewidth=1.8,
            markersize=5.5,
            capsize=3,
            elinewidth=1.0,
            label=variant,
        )

    ax.set_title(title, loc="left", fontsize=13, fontweight="semibold", pad=7)
    ax.set_xlabel("Excitation frequency (Hz)")
    ax.set_ylabel("Maximum amplitude (V)")

    ax.set_xticks([40, 120, 200, 280, 360, 440, 480])

    ax.legend(frameon=False, loc="upper right")

    finish_axis(ax)

axes[0].set_ylim(0, 1.34)
axes[1].set_ylim(-0.014, 0.064)

fig.tight_layout()

save_figure(fig, "figure_11_coated_uncoated")

plt.show()

### SI-2.3 Acoustic input across the tested frequencies

#### Figure SI-5 — Detection quality across channels and external acoustic input

In [ ]:
def midpoint(low, high):
    return (low + high) / 2.0


dbc_reference_df = pd.DataFrame(
    {
        "expected_frequency_Hz": [10, 60, 110, 160, 210, 260, 310, 360, 410, 460],
        "MIC_sound_level_dBC": [
            midpoint(64.0, 67.2),
            midpoint(72.5, 73.0),
            85.8,
            96.0,
            95.2,
            95.2,
            97.8,
            96.9,
            95.8,
            93.8,
        ],
        "ACC_sound_level_dBC": [
            midpoint(63.8, 66.1),
            midpoint(72.5, 72.8),
            85.5,
            91.1,
            95.8,
            95.0,
            97.1,
            96.8,
            96.3,
            92.8,
        ],
    }
)

dbc_reference_df["MIC_sound_reference_linear"] = 10 ** (
    dbc_reference_df["MIC_sound_level_dBC"] / 20.0
)

dbc_reference_df["ACC_sound_reference_linear"] = 10 ** (
    dbc_reference_df["ACC_sound_level_dBC"] / 20.0
)

ACC_DIAG_FILE = (
    FREQUENCY_RESPONSE_DIR
    / "FFT ACC"
    / "quantitative_diagnostics"
    / "acc_quantitative_frequency_diagnostics_channels_1_to_8.csv"
)

MIC_DIAG_FILE = (
    FREQUENCY_RESPONSE_DIR
    / "FFT MIC"
    / "quantitative_diagnostics"
    / "mic_quantitative_frequency_diagnostics_channels_1_to_8.csv"
)

for diagnostic_file in [ACC_DIAG_FILE, MIC_DIAG_FILE]:
    print("Reading:", diagnostic_file)

    if not diagnostic_file.is_file():
        raise FileNotFoundError(f"Diagnostic CSV not found:\n{diagnostic_file}")


def load_snr_diagnostics(file_path, sensor_type):
    data = pd.read_csv(file_path).copy()

    required_columns = {"expected_frequency_Hz", "channel", "target_snr_dB"}

    missing_columns = required_columns - set(data.columns)

    if missing_columns:
        raise KeyError(f"{file_path.name} is missing columns: " f"{sorted(missing_columns)}")

    data["sensor_type"] = sensor_type

    data["channel"] = data["channel"].astype(str).str.extract(r"(\d+)", expand=False)

    data["channel"] = pd.to_numeric(data["channel"], errors="coerce")

    data["expected_frequency_Hz"] = pd.to_numeric(
        data["expected_frequency_Hz"], errors="coerce"
    )

    data["target_snr_dB"] = pd.to_numeric(data["target_snr_dB"], errors="coerce")

    data = data.dropna(subset=["channel", "expected_frequency_Hz", "target_snr_dB"]).copy()

    data = data[data["channel"].between(1, 8)].copy()

    data["channel"] = data["channel"].astype(int)

    return data


mic_snr_raw = load_snr_diagnostics(MIC_DIAG_FILE, "MIC")

acc_snr_raw = load_snr_diagnostics(ACC_DIAG_FILE, "ACC")

snr_diagnostic_df = pd.concat([mic_snr_raw, acc_snr_raw], ignore_index=True)

channel_snr_df = snr_diagnostic_df.groupby(
    ["sensor_type", "expected_frequency_Hz", "channel"], as_index=False
).agg(target_snr_dB=("target_snr_dB", "mean"))

snr_summary_df = channel_snr_df.groupby(
    ["sensor_type", "expected_frequency_Hz"], as_index=False
).agg(
    n_channels=("channel", "nunique"),
    mean_target_snr_dB=("target_snr_dB", "mean"),
    sd_target_snr_dB=("target_snr_dB", "std"),
)

snr_summary_df["sem_target_snr_dB"] = snr_summary_df["sd_target_snr_dB"] / np.sqrt(
    snr_summary_df["n_channels"]
)

invalid_channel_counts = snr_summary_df[snr_summary_df["n_channels"] != 8]

if not invalid_channel_counts.empty:
    display(invalid_channel_counts)

    raise ValueError("Some modality-frequency groups do not contain all eight channels.")

display(
    snr_summary_df.pivot(
        index="expected_frequency_Hz", columns="sensor_type", values="mean_target_snr_dB"
    ).round(1)
)

fig, axes = plt.subplots(1, 2, figsize=(11.8, 4.6))

for sensor_type, color, label in [
    ("MIC", PRIMARY_COLOR, "MIC"),
    ("ACC", SECONDARY_COLOR, "ACC"),
]:
    data = snr_summary_df[snr_summary_df["sensor_type"] == sensor_type].sort_values(
        "expected_frequency_Hz"
    )

    x = data["expected_frequency_Hz"].to_numpy()
    y = data["mean_target_snr_dB"].to_numpy()
    error = data["sem_target_snr_dB"].to_numpy()

    axes[0].plot(x, y, marker="o", color=color, label=label)

    axes[0].fill_between(x, y - error, y + error, color=color, alpha=0.16, linewidth=0)

axes[0].set_title(
    "(a) Detection quality across channels",
    loc="left",
    fontsize=13,
    fontweight="semibold",
    pad=7,
)

axes[0].set_xlabel("Excitation frequency (Hz)")

axes[0].set_ylabel("Mean target-frequency SNR (dB)")

axes[0].set_xticks(dbc_reference_df["expected_frequency_Hz"])

axes[0].tick_params(axis="x", rotation=35)

maximum_snr_with_sem = (
    snr_summary_df["mean_target_snr_dB"] + snr_summary_df["sem_target_snr_dB"]
).max()

axes[0].set_ylim(0, maximum_snr_with_sem * 1.10)

axes[0].legend(frameon=False, loc="upper right")

finish_axis(axes[0])

axes[1].plot(
    dbc_reference_df["expected_frequency_Hz"],
    dbc_reference_df["MIC_sound_level_dBC"],
    marker="o",
    color=PRIMARY_COLOR,
    label="MIC run",
)

axes[1].plot(
    dbc_reference_df["expected_frequency_Hz"],
    dbc_reference_df["ACC_sound_level_dBC"],
    marker="o",
    color=SECONDARY_COLOR,
    label="ACC run",
)

axes[1].set_title(
    "(b) External acoustic input measured near the patch",
    loc="left",
    fontsize=13,
    fontweight="semibold",
    pad=7,
)

axes[1].set_xlabel("Excitation frequency (Hz)")
axes[1].set_ylabel("Sound-level reference (dBC)")

axes[1].set_xticks(dbc_reference_df["expected_frequency_Hz"])

axes[1].tick_params(axis="x", rotation=35)

axes[1].legend(frameon=False, loc="upper left")

finish_axis(axes[1])

fig.tight_layout()

save_figure(fig, "figure_13_detection_and_external_input")

plt.show()